# Kernex demo

Kernex evaluates symbolic expressions on CUDA and can interpret expression bytecode inline inside an existing Numba CUDA kernel.

In [ ]:
import numpy as np
from numba import cuda

from kernex import Expression, ExpressionVector
from kernex.expression.inline import evaluate_expression_inline

## Scalar expression

In [ ]:
expr = Expression(
    'H0*k1/(k2 + k3) - H1*k3/(k1 - k2) - k0',
    parameter_order=['H0', 'H1', 'k0', 'k1', 'k2', 'k3'],
)
parameters = np.random.uniform(0, 1, (100_000, 6))
gpu_result = expr.evaluate(parameters)

## Inline evaluation inside a kernel

In [ ]:
inline_expr = Expression(
    'k1*x/(k2 + x)',
    parameter_order=['k1', 'k2'],
    variable_order=['x'],
)
device_bytecode = inline_expr.to_device()

@cuda.jit
def kernel(bytecode, params, variables, workspace, results):
    i = cuda.grid(1)
    if i < params.shape[0]:
        results[i] = evaluate_expression_inline(
            bytecode, params[i], variables[i], workspace[i]
        )

## NumPy reference backend

In [ ]:
reference = expr.to_numpy()
numpy_result = reference(parameters)
np.allclose(gpu_result, numpy_result)

## Vector expressions

In [ ]:
vector_expr = ExpressionVector(
    ['k4/(k1*k2)', 'k1/k4+k3/k2', 'k1+k2/(k1-k3)', 'k4+k2'],
    parameter_order=['k1', 'k2', 'k3', 'k4'],
)
parameters = np.random.uniform(0, 1, (100_000, 4))
vector_result = vector_expr.evaluate(parameters)